# Exercises on Information Retrieval

<img src="images/gandalf.jpg" width="600" />

## What is Information Retrieval?

- The process of obtaining relevant information from a large repository of data.
- Involves finding material (usually documents) that satisfies an information need.

So, there are already three important concepts:
- documents as unit of information
- queries: the user need
- relevance: a measure of how well a document meets the user need


## Very Brief History of Information Retrieval

- Shannon's Information Theory (1948):
    Claude Shannon's groundbreaking work on information theory laid the foundation for understanding how information can be quantified and transmitted.

- 1960s: Introduced vector space model, term frequency-inverse document frequency (TF-IDF), and relevance feedback mechanisms.

- 1990s: Rise of the World Wide Web
    - Search Engines
    - PageRank Algorithm (1996):
- 2000s: 
    - Integration of machine learning techniques and natural language processing (NLP) for more accurate and context-aware retrieval. 
    - Personalization and Recommendation Systems using user data.

Today we'll do a few simple information retrieval operations for searching over a sample of British Library books

In [ ]:
!pip install wget

In [ ]:
#download dataset using wget store it data folder
#url: https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/dev/Sessions/data/bl_books_sample.csv

import wget
import os

url = 'https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/dev/Sessions/data/bl_books_sample.csv'
filename = url.split('/')[-1]

if not os.path.exists('data'):
    os.makedirs('data')

filepath = os.path.join('data', filename)
if os.path.exists(filepath):
    print(f'{filepath} already exists')
else:
    wget.download(url, filepath)
    print(f'Downloaded {filename} to {filepath}')


In [ ]:
import pandas as pd

# Here we load a sample of 1000 books from a British Library dataset
df = pd.read_csv('data/bl_books_sample.csv')

len(df)

In [ ]:
# Let's see the first few rows
df.head()

### ✏️ Exercise 1

Search a single term in the text of the articles and retrieve the top 5 articles, ranked based on the frequency of that term

### Issue: Longer documents are ranked higher

How do we address this?


<img src="images/long_doc.jpg" width="300" />

### ✏️ Exercise 2

Normalise word counts by the length of the document

### How to search for multiple keywords?

We have multiple options:
- searching for a phrase
- searching for a series of keywords and combine the counts


### ✏️ Exercise 3
Search an entire phrase

### ✏️ Exercise 4

Search multiple terms at once weighting them for the length of the documents

### Issue: Some words are more important than others, but how which ones?

A solution for our problems: TF-IDF

<img src="images/tfidf.png"  />

### ✏️ Exercise 5 (Advanced)
Normalise word counts by using TF-IDF and retrieve the top five articles given a query

## Exercise for Thursdays

Issue: TF-IDF does not capture semantic information, only frequency. Can we use word embeddings instead?

### ✏️ Exercise 6
Transform each text in a document embedding and then find the 5 most similar to the query

## Solutions!

In [ ]:
# Exercise 1
term = 'war'
df['count'] = df.text.str.count(term)
df = df.sort_values('count', ascending=False)
print(df[['text', 'count']].head(5))

In [ ]:
# Exercise 2
df['norm_count'] = df['count'] / df['text'].str.len()
df = df.sort_values('norm_count', ascending=False)
print(df[['text', 'count','norm_count']].head(5))

In [ ]:
# Exercise 3
terms = 'the elephant is the most gentle'
df['count'] = df.text.str.count(terms)
df['norm_count'] = df['count'] / df['text'].str.len()
df = df.sort_values('norm_count', ascending=False)
print(df[['text', 'count','norm_count']].head(5))

In [ ]:
# Exercise 4
terms = ['the','power', 'of','democracy']
df['count'] = df.text.str.count('|'.join(terms))
df['norm_count'] = df['count'] / df['text'].str.len()
df = df.sort_values('norm_count', ascending=False)
print(df[['text', 'count','norm_count']].head(5))

In [ ]:
# Exercise 5

## given a term, find the top 5 articles that contain the term based on TFIDF
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize a TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english')

# Fit and transform the 'text' data
tfidf_matrix = tfidf_vectorizer.fit_transform(df['text'])

# Convert the term "power" into its TF-IDF representation
query_vector = tfidf_vectorizer.transform(["the power of democracy"])

# Compute similarity scores
from sklearn.metrics.pairwise import cosine_similarity
similarity_scores = cosine_similarity(query_vector, tfidf_matrix)

# Get top 5 article indices based on similarity scores
top_article_indices = similarity_scores.argsort()[0][-5:]

# Retrieve top 5 articles
top_articles = df.iloc[top_article_indices]

print(top_articles.iloc[0].text)

In [ ]:
# Exercise 6

# convert each text to a document embedding using spacy
import spacy
spacy.cli.download("en_core_web_sm")

nlp = spacy.load('en_core_web_sm')

# Create a function to convert text to a document embedding
def get_embedding(text):
    return nlp(text).vector

# Apply the function to the 'text' column
df['embedding'] = df['text'].apply(get_embedding)

# Convert the query to a document embedding
query_embedding = get_embedding("the power of democracy")

# Compute similarity scores
from sklearn.metrics.pairwise import cosine_similarity
df['emb_similarity'] = df['embedding'].apply(lambda x: cosine_similarity([query_embedding], [x])[0][0])

# Get top 5 articles based on similarity scores
top_articles = df.sort_values('emb_similarity', ascending=False).head(5)

print(top_articles[['text', 'emb_similarity']])
print(top_articles.iloc[0].text)